# Maintenance Work Order Analysis with Ollama

This notebook reads the sample maintenance work order CSV, calls a local Ollama model, and extracts the likely equipment and failure mode from each work order description.

Recommended starting model for a Dell Inspiron 15 5510: `qwen2.5:3b`. It is small enough for typical CPU-only laptop use while still being capable at structured extraction. If the laptop has 16 GB RAM and you can tolerate slower runs, try `qwen2.5:7b`. If it has 8 GB RAM or feels sluggish, use `llama3.2:3b` or `gemma3:1b`.

## One-time setup

Install Ollama, then pull the model in a terminal:

```bash
ollama pull qwen2.5:3b
ollama serve
```

Install notebook dependencies if needed:

```bash
python3 -m pip install -r requirements.txt
```

In [ ]:
from pathlib import Path
import json
import re

import pandas as pd
import requests

In [ ]:
CSV_PATH = Path("../data/maintenance_work_orders.csv")
if not CSV_PATH.exists():
    CSV_PATH = Path("data/maintenance_work_orders.csv")

OUTPUT_PATH = CSV_PATH.with_name("maintenance_work_orders_analysed.csv")

OLLAMA_URL = "http://localhost:11434/api/chat"
MODEL = "qwen2.5:3b"

df = pd.read_csv(CSV_PATH)
df.head()

In [ ]:
def check_ollama_available() -> None:
    try:
        response = requests.get("http://localhost:11434/api/tags", timeout=5)
        response.raise_for_status()
    except requests.RequestException as exc:
        raise RuntimeError(
            "Ollama is not reachable. Start it with `ollama serve` and make sure the model is pulled."
        ) from exc


check_ollama_available()

In [ ]:
SYSTEM_PROMPT = """
You are a maintenance reliability analyst. Extract structured failure information from work order text.

Return only valid JSON with these exact keys:
- equipment_failed: the specific asset or equipment mentioned, such as pump P-204 or conveyor CV-101.
- failure_mode: concise physical or functional failure mode, such as mechanical seal leak, motor overload trip, bearing wear, sensor false trigger, blocked suction strainer, no failure - preventive maintenance.
- confidence: high, medium, or low.
- evidence: short phrase from the work order that supports the answer.

Rules:
- If breakdown_type is preventive and the description does not report a fault, set failure_mode to "no failure - preventive maintenance".
- Do not invent equipment that is not present in the description.
- Keep each value short and practical for maintenance analysis.
""".strip()


def build_user_prompt(row: pd.Series) -> str:
    return f"""
Work order number: {row.work_order_number}
Date created: {row.date_created}
Breakdown type: {row.breakdown_type}
Status: {row.work_order_status}
Description: {row.description}
""".strip()

In [ ]:
def parse_json_response(text: str) -> dict:
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if not match:
            raise
        return json.loads(match.group(0))


def analyse_work_order(row: pd.Series) -> dict:
    payload = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_user_prompt(row)},
        ],
        "format": "json",
        "stream": False,
        "options": {"temperature": 0},
    }

    response = requests.post(OLLAMA_URL, json=payload, timeout=120)
    response.raise_for_status()
    content = response.json()["message"]["content"]
    parsed = parse_json_response(content)

    return {
        "equipment_failed": parsed.get("equipment_failed", "unknown"),
        "failure_mode": parsed.get("failure_mode", "unknown"),
        "llm_confidence": parsed.get("confidence", "low"),
        "llm_evidence": parsed.get("evidence", ""),
    }

In [ ]:
sample_result = analyse_work_order(df.iloc[0])
sample_result

In [ ]:
results = []

for index, row in df.iterrows():
    print(f"Analysing {row.work_order_number} ({index + 1}/{len(df)})")
    try:
        result = analyse_work_order(row)
    except Exception as exc:
        result = {
            "equipment_failed": "error",
            "failure_mode": "error",
            "llm_confidence": "low",
            "llm_evidence": str(exc),
        }
    results.append(result)

llm_columns = ["equipment_failed", "failure_mode", "llm_confidence", "llm_evidence"]
llm_results_df = pd.DataFrame(results).reindex(columns=llm_columns)

analysis_df = pd.concat([df, llm_results_df], axis=1)
analysis_df.head(10)

In [ ]:
analysis_df.to_csv(OUTPUT_PATH, index=False)
OUTPUT_PATH

In [ ]:
analysis_df.groupby(["breakdown_type", "failure_mode"]).size().reset_index(name="count").sort_values(
    ["breakdown_type", "count"], ascending=[True, False]
)